# 02d — 第二讲基础习题：回归、交叉验证与分类巩固练习

本套习题帮助你一步步巩固第二讲（`02a`–`02c`）中使用 `scikit-learn` 构建监督学习模型的标准操作。每道题只需要填写 **2–5 行代码**，并附有预期数值提示供你自查。完成后可对照 `solutions/02_exercises_solutions.ipynb` 查看完整解答。

---

## 0. 公共环境与 ESOL 特征矩阵准备

请先运行下方单元格，读入 ESOL 数据集并计算好 10 个 RDKit 物理化学描述符特征矩阵 `X`（形状 `1128 × 10`）以及连续目标向量 `y_reg`（实测 `logS`）。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Descriptors

for p in [Path("../01 - Basics/data/esol.csv"), Path("data/esol.csv")]:
    if p.exists():
        df = pd.read_csv(p)
        break

df = df.rename(columns={
    "Compound ID": "name",
    "measured log solubility in mols per litre": "logS",
})[["name", "smiles", "logS"]].copy()
df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)
df = df.dropna(subset=["mol"]).reset_index(drop=True)

DESC_FUNCS = {
    "MolWt": Descriptors.MolWt,
    "MolLogP": Descriptors.MolLogP,
    "TPSA": Descriptors.TPSA,
    "NumHDonors": Descriptors.NumHDonors,
    "NumHAcceptors": Descriptors.NumHAcceptors,
    "NumRotatableBonds": Descriptors.NumRotatableBonds,
    "RingCount": Descriptors.RingCount,
    "NumAromaticRings": Descriptors.NumAromaticRings,
    "HeavyAtomCount": Descriptors.HeavyAtomCount,
    "FractionCSP3": Descriptors.FractionCSP3,
}
X = pd.DataFrame([{k: fn(m) for k, fn in DESC_FUNCS.items()} for m in df["mol"]])
y_reg = df["logS"].values

print(f"成功准备特征矩阵 X: {X.shape} | 目标向量 y_reg: {y_reg.shape}")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---

## 第一部分：回归基础与特征缩放（对应 `02a`）

### 🏋️ 习题 1：训练/测试集划分与“均值猜测基线”

1. 使用 `train_test_split` 将 `X` 和 `y_reg` 按 **测试集占 25%（`test_size=0.25`）**、随机种子 `random_state=0` 划分为 `X_train, X_test, y_train, y_test`；
2. 假设有一个完全不看分子特征的傻瓜基线，对测试集每个分子都预测为训练集均值 `y_train.mean()`，请计算该基线在 `y_test` 上的 `MAE` 和 `RMSE`。
- **预期输出**：`Baseline MAE = 1.690 | Baseline RMSE = 2.088`

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 在下方填写你的代码：


### 🏋️ 习题 2：训练多元线性回归并计算测试集 $R^2$ 与 RMSE

使用上一题划分好的 `X_train, y_train`：
1. 实例化并拟合一个多元线性回归模型 `LinearRegression()`；
2. 对 `X_test` 进行预测，打印测试集上的 `RMSE` 和决定系数 `R2`（保留 3 位小数）。
- **预期输出**：`LinearRegression Test RMSE = 0.903 | Test R2 = 0.812`

In [ ]:
from sklearn.linear_model import LinearRegression

# 在下方填写你的代码：


### 🏋️ 习题 3：验证特征标准化对 k 近邻回归（kNN）的提升

1. 直接在原始特征 `X_train` 上训练 `KNeighborsRegressor(n_neighbors=5)`，计算其在 `X_test` 上的测试 RMSE `rmse_raw`；
2. 用 `StandardScaler()` 在 `X_train` 上 `fit_transform`，并对 `X_test` 执行 `transform`；在标准化后的特征上重新训练 `KNeighborsRegressor(n_neighbors=5)`，计算测试 RMSE `rmse_scaled`。
- **预期输出**：`未标准化 RMSE = 0.931 | 标准化后 RMSE = 0.727`

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

# 在下方填写你的代码：


---

## 第二部分：防泄漏 Pipeline、交叉验证与超参数调优（对应 `02b`）

### 🏋️ 习题 4：用 `make_pipeline` + `cross_val_score` 执行 5 折交叉验证

为了防止在交叉验证中发生数据泄漏：
1. 用 `make_pipeline(StandardScaler(), Ridge(alpha=1.0))` 创建一个包含标准化和岭回归的流水线 `ridge_pipe`；
2. 调用 `cross_val_score(ridge_pipe, X, y_reg, cv=5, scoring="r2")` 计算它在全数据集 5 折交叉验证下的决定系数 $R^2$，打印 5 折的均值与标准差。
- **预期输出**：`5-Fold CV R2: 0.799 ± 0.025`

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# 在下方填写你的代码：


### 🏋️ 习题 5：使用 `GridSearchCV` 为单棵决策树寻找最优深度

单棵决策树如果深度不加限制（`max_depth=None`）非常容易过拟合。
1. 对 `DecisionTreeRegressor(random_state=42)` 使用 `GridSearchCV`，在参数网格 `param_grid = {"max_depth": [2, 4, 6, 8, 12, None]}` 上进行 5 折交叉验证搜索（评分标准设为 `scoring="neg_root_mean_squared_error"`），在 `X_train, y_train` 上拟合；
2. 打印搜索到的最优参数 `grid.best_params_`，以及该最优模型在测试集 `X_test` 上的 RMSE。
- **预期输出**：`最优参数: {'max_depth': 6} | 测试集 RMSE = 0.822`

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

param_grid = {"max_depth": [2, 4, 6, 8, 12, None]}

# 在下方填写你的代码：


---

## 第三部分：二元分类与特征重要性（对应 `02c`）

### 🏋️ 习题 6：构造二分类标签并进行分层抽样划分（`stratify`）

假设某流体加工工艺要求分子水溶解度至少达到 $\log S \ge -3.0$，我们将 **$\log S < -3.0$ 的分子定义为需要拦截的正类（`1`）**，$\log S \ge -3.0$ 定义为负类（`0`）：
1. 生成整数标签数组 `y_cls = (df["logS"] < -3.0).astype(int)`，打印正类（`1`）在全集中的占比；
2. 使用 `train_test_split`（设 `test_size=0.2, stratify=y_cls, random_state=42`）将 `X, y_cls` 划分为 `X_tr_c, X_te_c, y_tr_c, y_te_c`，验证训练集与测试集的正类占比是否一致。
- **预期输出**：正类占比约 `48.2%`（训练集 `0.482`，测试集 `0.482`）

In [ ]:
# 在下方填写你的代码：


### 🏋️ 习题 7：训练逻辑回归分类器并计算核心分类指标

在上一题划分好的 `X_tr_c, y_tr_c` 上：
1. 用 `make_pipeline(StandardScaler(), LogisticRegression())` 训练逻辑回归分类器 `clf_lr`；
2. 在测试集 `X_te_c, y_te_c` 上计算并打印 `Accuracy`、`Precision`、`Recall`、`F1` 以及 `ROC-AUC`（注意 `roc_auc_score` 需要传入 `clf_lr.predict_proba(X_te_c)[:, 1]`）。
- **预期输出**：`Accuracy = 0.858 | Precision = 0.847 | Recall = 0.862 | F1 = 0.855 | ROC-AUC = 0.941`

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 在下方填写你的代码：


### 🏋️ 习题 8：训练随机森林分类器并绘制特征重要性条形图

1. 在 `X_tr_c, y_tr_c` 上训练 `RandomForestClassifier(n_estimators=100, random_state=42)`；
2. 提取模型的 `feature_importances_` 属性，封装为以 `X.columns` 为索引的 `pd.Series` 并按从小到大排序（`.sort_values()`）；
3. 调用 `.plot(kind="barh")` 画出水平条形图，观察排名前三的特征。

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 在下方填写你的代码：
